# Problema resuelto 23: Calculo Funcional y Derivadas Funcionales

Cuaderno de apoyo para `Tutorial/00_prerrequisitos/06_calculo_funcional_y_derivadas.md`

**Objetivo:** verificar con SymPy las reglas de derivacion funcional, calcular ecuaciones de movimiento a partir de acciones y explorar el funcional generador de una teoria libre.

In [ ]:
import sympy as sp

x, y, m, phi_val = sp.symbols('x y m phi_val', real=True)
phi = sp.Function('phi')
J   = sp.Function('J')

## Problema 1

**Resultado basico:** $\dfrac{\delta\phi(x)}{\delta\phi(y)} = \delta^{(4)}(x-y)$.

En el limite discreto, si $\phi_i$ son los valores del campo en una red y $\delta_{ij}$ es el Kronecker, entonces:

$$\frac{\partial \phi_i}{\partial \phi_j} = \delta_{ij}.$$

Al pasar al continuo, $\delta_{ij}$ se convierte en $\delta^{(4)}(x-y)$. Verifiquemos el caso discreto.

In [ ]:
# Caso discreto: campo en una red de 4 puntos
phi_vec = sp.Matrix(sp.symbols('phi_1 phi_2 phi_3 phi_4'))

# La derivada parcial phi_i respecto a phi_j es exactamente delta_ij
jacobian = phi_vec.jacobian(phi_vec)
print("Jacobiano (analogo discreto de delta funcional):")
sp.pprint(jacobian)

### Resultado

El jacobiano es la identidad: $\partial\phi_i/\partial\phi_j = \delta_{ij}$. En el limite continuo, esto se convierte en $\delta\phi(x)/\delta\phi(y) = \delta^{(4)}(x-y)$.

## Problema 2

**Regla del producto funcional.** Dado el funcional $F[\phi] = \phi(x)^2$, calcular

$$\frac{\delta F}{\delta\phi(y)} = \frac{\delta}{\delta\phi(y)}\phi(x)^2.$$

Usando la regla del producto: $\delta(\phi^2) = 2\phi\,\delta\phi$, y como $\delta\phi(x)/\delta\phi(y)=\delta^{(4)}(x-y)$:

$$\frac{\delta\phi(x)^2}{\delta\phi(y)} = 2\phi(y)\,\delta^{(4)}(x-y).$$

Integrando sobre $x$:

In [ ]:
# Ilustracion simbolica en el caso discreto
phi1, phi2 = sp.symbols('phi_1 phi_2', real=True)

F = phi1**2 + phi2**2  # analogo discreto de integral de phi^2

dF_dphi1 = sp.diff(F, phi1)
dF_dphi2 = sp.diff(F, phi2)

print(f"dF/dphi_1 = {dF_dphi1}  (analogo: 2*phi(y)*delta(x-y) integrado)")
print(f"dF/dphi_2 = {dF_dphi2}")

## Problema 3

**Ecuacion de Klein-Gordon a partir de la accion.** La accion del campo escalar libre es

$$S[\phi] = \int d^4x\left(\frac{1}{2}(\partial_\mu\phi)^2 - \frac{1}{2}m^2\phi^2\right).$$

Aplicando las ecuaciones de Euler-Lagrange a $\mathcal{L} = \frac{1}{2}(\partial_\mu\phi)^2 - \frac{1}{2}m^2\phi^2$:

$$\frac{\partial\mathcal{L}}{\partial\phi} - \partial_\mu\frac{\partial\mathcal{L}}{\partial(\partial_\mu\phi)} = 0.$$

In [ ]:
t = sp.Symbol('t', real=True)
phi_t = sp.Function('phi')(t)  # campo en 1+0 dimensiones para ilustrar
m_sym = sp.Symbol('m', positive=True)

dphi = sp.diff(phi_t, t)

# Lagrangiano en 0+1d: L = (1/2)*phi_dot^2 - (1/2)*m^2*phi^2
L = sp.Rational(1,2)*dphi**2 - sp.Rational(1,2)*m_sym**2*phi_t**2

dL_dphi    = sp.diff(L, phi_t)
dL_ddphi   = sp.diff(L, dphi)
EL = dL_dphi - sp.diff(dL_ddphi, t)

print("Ecuacion de Euler-Lagrange:")
sp.pprint(sp.simplify(EL), use_unicode=True)
print("\n=> Ecuacion del oscilador: phi'' + m^2*phi = 0")
print("   En 3+1d esto generaliza a: ([] + m^2)phi = 0  (Klein-Gordon)")

## Problema 4

**Funcional generador de teoria libre.** Para la teoria libre con fuente $J$:

$$Z[J] = Z_0\,\exp\!\left(-\frac{1}{2}\int d^4x\,d^4y\, J(x)\,D_F(x-y)\,J(y)\right),$$

donde $D_F(x-y)$ es el propagador de Feynman. El correlador de dos puntos es:

$$\langle\phi(x)\phi(y)\rangle = \frac{1}{Z_0}\frac{\delta^2 Z}{\delta J(x)\delta J(y)}\bigg|_{J=0} = D_F(x-y).$$

Ilustramos esto con el analogo de una integral gaussiana.

In [ ]:
import numpy as np

# Analogo de dimension finita: Z[j] = exp(j^T A^{-1} j / 2)
# con A = masa^2 * identidad (teoria libre)

mass = 1.0
N    = 4  # numero de puntos de la red
A    = mass**2 * np.eye(N)
Ainv = np.linalg.inv(A)

# Propagador libre en la red: Ainv[i,j] = delta_ij / m^2
print("Propagador libre (analogo discreto de D_F):")
print(np.round(Ainv, 3))
print(f"\nTodos los terminos diagonales = 1/m^2 = {1/mass**2:.3f}")
print("Los terminos fuera de la diagonal son cero: no hay correlaciones en teoria libre")

## Problema 5

**Verificacion de la regla de la cadena funcional.** Si $F[\phi] = G[H[\phi]]$ entonces:

$$\frac{\delta F}{\delta\phi(y)} = \int d^4z\, \frac{\delta G}{\delta H(z)}\,\frac{\delta H(z)}{\delta\phi(y)}.$$

Aplicacion concreta: $H[\phi] = \phi^2(x)$, $G[H] = H^2$, entonces $F[\phi] = \phi^4(x)$.

In [ ]:
phi_sym = sp.Symbol('phi', real=True)

# F = phi^4
F4 = phi_sym**4

# Derivada directa
dF4 = sp.diff(F4, phi_sym)

# Via regla de la cadena: G=H^2, H=phi^2
H   = phi_sym**2
G_H = H**2
dG_dH = sp.diff(G_H, H)     # 2H
dH_dphi = sp.diff(H, phi_sym)  # 2*phi
chain = dG_dH * dH_dphi

print(f"Derivada directa de phi^4: {dF4}")
print(f"Via regla cadena (dG/dH * dH/dphi): {sp.expand(chain)}")
print(f"Son iguales: {sp.simplify(dF4 - chain) == 0}")

## Resumen

| Concepto | Formula clave |
|---|---|
| Resultado basico | $\delta\phi(x)/\delta\phi(y) = \delta^{(4)}(x-y)$ |
| Regla del producto | $\delta(FG)/\delta\phi = (\delta F/\delta\phi)G + F(\delta G/\delta\phi)$ |
| Euler-Lagrange | $\delta S/\delta\phi = 0$ da las ecuaciones de movimiento |
| Correlador libre | $\langle\phi(x)\phi(y)\rangle = D_F(x-y) = \delta^2 Z/\delta J(x)\delta J(y)|_{J=0}$ |

Estos cuatro resultados cubren la mayor parte del calculo funcional que aparece en los modulos centrales del tutorial.